# Machine Learning Approach for Liver Cancer Patients Classification

## Project Overview

This notebook implements and evaluates multiple machine learning classification models to predict liver cancer. The workflow follows standard data science best practices:

1. **Data Loading & Exploration** – Load the dataset and inspect its structure, types, and basic statistics.
2. **Data Preprocessing** – Handle categorical encoding, outlier detection, feature scaling, and feature selection.
3. **Model Training** – Train five classification models with reproducible random seeds.
4. **Model Evaluation** – Compare models using accuracy, precision, recall, F1-score, confusion matrices, ROC curves, and cross-validation.
5. **Best Model Selection** – Identify the best-performing model based on evaluation metrics.

## Models Evaluated
- Logistic Regression
- Random Forest Classifier
- Support Vector Machine (SVM)
- K-Nearest Neighbors (KNN)
- XGBoost Classifier

In [ ]:
# Standard library
import warnings
import math
from io import StringIO

# Data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats

# Scikit-learn – preprocessing & utilities
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.feature_selection import SelectKBest, f_classif

# Scikit-learn – classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Scikit-learn – metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
)

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

In [ ]:
# Place the CSV file in the same directory as this notebook and update the path below if needed.
DATA_PATH = 'synthetic_liver_cancer_dataset.csv'

try:
    df = pd.read_csv(DATA_PATH)
    print(f'Dataset loaded successfully. Shape: {df.shape}')
except FileNotFoundError:
    raise FileNotFoundError(
        f'Dataset not found at "{DATA_PATH}". '
        'Please place the CSV file in the same directory as this notebook.'
    )

In [ ]:
df.head()  # Preview first five rows

In [ ]:
# Encode ordinal and binary categorical columns with meaningful integer mappings.

# Binary: Gender
df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})

# Ordinal: Alcohol consumption (ordered by intensity)
df['alcohol_consumption'] = df['alcohol_consumption'].map({
    'Never': 0,
    'Occasional': 1,
    'Regular': 2,
})

# Ordinal: Smoking status (ordered by exposure level)
df['smoking_status'] = df['smoking_status'].map({
    'Never': 0,
    'Former': 1,
    'Current': 2,
})

# Ordinal: Physical activity level (ordered by activity intensity)
df['physical_activity_level'] = df['physical_activity_level'].map({
    'Low': 0,
    'Moderate': 1,
    'High': 2,
})

print('Encoding complete. Data types after encoding:')
print(df.dtypes)

# Descriptive Statistics

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values found.')

In [ ]:
# Summary statistics for all numeric columns
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# Target class distribution
print('Target variable distribution (liver_cancer):')
print(df['liver_cancer'].value_counts())
print(f'Class balance ratio: {df["liver_cancer"].value_counts(normalize=True).round(3).to_dict()}')

# Feature correlations with the target variable
print('\nPearson correlations with liver_cancer (sorted):')
print(df.corr()['liver_cancer'].sort_values(ascending=False))

In [ ]:
# Inspect column data types and non-null counts
buf = StringIO()
df.info(buf=buf)
print(buf.getvalue())

# Data Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['age'], kde=True, color='steelblue', ax=ax)
ax.set_title('Age Distribution of Patients')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
corr_target = df.corr()['liver_cancer'].drop('liver_cancer').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(6, 8))
sns.heatmap(
    corr_target.to_frame(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title('Feature Correlation with Liver Cancer')
plt.tight_layout()
plt.show()

In [ ]:
# Full correlation matrix heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    df.corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title('Full Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Step 1: Univariate Analysis – Distribution of all numerical features
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
n_cols = len(numerical_cols)
n_plot_cols = 4
n_plot_rows = math.ceil(n_cols / n_plot_cols)

fig, axes = plt.subplots(n_plot_rows, n_plot_cols, figsize=(16, 4 * n_plot_rows))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.histplot(df[col], kde=True, color='steelblue', bins=20, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

# Hide any unused subplot axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Step 1: Univariate Analysis (Distributions)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Step 2: Feature Relationships – Pairplot of numerical features
# Note: limit to a subset of columns if the dataset is large to avoid slow rendering.
key_features = ['age', 'bmi', 'liver_cancer']  # Adjust as needed
pairplot_cols = [c for c in key_features if c in df.columns]

pair_grid = sns.pairplot(
    df[pairplot_cols],
    hue='liver_cancer' if 'liver_cancer' in pairplot_cols else None,
    diag_kind='kde',
    plot_kws={'alpha': 0.5},
)
pair_grid.figure.suptitle('Step 2: Pair Plot of Key Numerical Features', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Step 3: Outlier Detection
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
n_cols = len(numerical_cols)
n_plot_cols = 4
n_plot_rows = math.ceil(n_cols / n_plot_cols)

# --- Boxplots ---
fig, axes = plt.subplots(n_plot_rows, n_plot_cols, figsize=(16, 4 * n_plot_rows))
axes = axes.flatten()
for i, col in enumerate(numerical_cols):
    sns.boxplot(x=df[col], color='salmon', ax=axes[i])
    axes[i].set_title(f'Boxplot – {col}')
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Outlier Detection: Boxplots', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# --- IQR Method ---
print('Outliers detected via IQR method:')
for col in numerical_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    print(f'  {col}: {n_outliers} outliers')

# --- Z-Score Method ---
print('\nOutliers detected via Z-score method (|z| > 3):')
z_threshold = 3
for col in numerical_cols:
    z_scores = np.abs(stats.zscore(df[col], nan_policy='omit'))
    n_outliers = (z_scores > z_threshold).sum()
    print(f'  {col}: {n_outliers} outliers')

In [ ]:
# Apply log1p transformation to BMI to reduce right skewness.
# log1p(x) = log(1 + x) is safe for values >= 0 and handles zeros gracefully.
df['bmi'] = np.log1p(df['bmi'])
print('BMI log-transformation applied.')
print(df['bmi'].describe())

In [ ]:
# Verify class balance of the binary target variable.
# Note: capping a binary (0/1) target variable is not appropriate for classification.
class_counts = df['liver_cancer'].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot(kind='bar', color=['steelblue', 'tomato'], ax=ax)
ax.set_title('Target Class Distribution')
ax.set_xlabel('Liver Cancer (0 = No, 1 = Yes)')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', va='bottom')
plt.tight_layout()
plt.show()
print(f'Class balance: {class_counts.to_dict()}')

# Data Preprocessing for Liver Cancer Prediction

In this phase we prepare the dataset for model training using these steps:

1. **Train/test split** – 80 % training, 20 % testing, stratified by the target class.
2. **Feature scaling** – Standardise numerical features (zero mean, unit variance) using `StandardScaler` fitted on the training set only to prevent data leakage.
3. **Categorical encoding** – One-hot encoding for any remaining nominal features.
4. **Feature selection** – Select the top-*k* most relevant features using ANOVA F-statistic (`SelectKBest`).

In [ ]:
# Step 1: Split the data into training (80 %) and testing (20 %) sets.
# Stratify ensures both splits maintain the original class ratio.
X = df.drop('liver_cancer', axis=1)  # Feature matrix
y = df['liver_cancer']               # Target vector

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set size : {X_train.shape[0]} samples')
print(f'Testing set size  : {X_test.shape[0]} samples')
print(f'Training class balance : {y_train.value_counts().to_dict()}')
print(f'Testing class balance  : {y_test.value_counts().to_dict()}')

In [ ]:
# Step 2: Feature Scaling – Standardise numerical features.
# The scaler is fitted on training data only to avoid data leakage.
numerical_cols = X.select_dtypes(include=[float, int]).columns
categorical_cols = X.select_dtypes(include=[object]).columns

scaler = StandardScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# Step 3: One-Hot Encoding for any remaining categorical columns.
X_train_enc = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_enc = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align columns in case get_dummies creates different columns in train vs. test.
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)

# Step 4: Feature Selection – keep the top-10 features by ANOVA F-statistic.
k_best = min(10, X_train_enc.shape[1])  # Guard against fewer available features
selector = SelectKBest(score_func=f_classif, k=k_best)
X_train_selected = selector.fit_transform(X_train_enc, y_train)
X_test_selected = selector.transform(X_test_enc)

selected_features = X_train_enc.columns[selector.get_support()]
print(f'Selected {k_best} features: {selected_features.tolist()}')
print(f'Training set shape: {X_train_selected.shape}')
print(f'Testing set shape : {X_test_selected.shape}')

# Evaluation of Machine Learning Models for Liver Cancer Prediction

We train and evaluate five classifiers using:

| Metric | Description |
|--------|-------------|
| **Accuracy** | Proportion of correctly classified samples |
| **Precision** | TP / (TP + FP) — how many predicted positives are truly positive |
| **Recall** | TP / (TP + FN) — how many actual positives are correctly identified |
| **F1-Score** | Harmonic mean of precision and recall |
| **ROC-AUC** | Area under the Receiver Operating Characteristic curve |
| **CV Accuracy** | Mean 5-fold stratified cross-validation accuracy |

All models use `random_state=42` where applicable to ensure reproducibility.

In [ ]:
# Define classifiers with reproducible random seeds where available.
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Support Vector Machine': SVC(probability=True, random_state=RANDOM_STATE),
    # KNeighborsClassifier has no random_state; minor variability in tie-breaking is expected.
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'XGBoost': XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}
trained_models = {}

for name, model in models.items():
    # Train
    model.fit(X_train_selected, y_train)
    trained_models[name] = model

    # Predict
    y_pred = model.predict(X_test_selected)
    y_prob = (
        model.predict_proba(X_test_selected)[:, 1]
        if hasattr(model, 'predict_proba')
        else model.decision_function(X_test_selected)
    )

    # Metrics
    report = classification_report(y_test, y_pred, output_dict=True)
    cv_scores = cross_val_score(model, X_train_selected, y_train, cv=cv, scoring='accuracy')
    roc_auc = roc_auc_score(y_test, y_prob)

    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': report['weighted avg']['precision'],
        'recall': report['weighted avg']['recall'],
        'f1_score': report['weighted avg']['f1-score'],
        'roc_auc': roc_auc,
        'cv_accuracy_mean': cv_scores.mean(),
        'cv_accuracy_std': cv_scores.std(),
    }

    print(f'--- {name} ---')
    print(f'  Accuracy  : {results[name]["accuracy"]:.4f}')
    print(f'  Precision : {results[name]["precision"]:.4f}')
    print(f'  Recall    : {results[name]["recall"]:.4f}')
    print(f'  F1-Score  : {results[name]["f1_score"]:.4f}')
    print(f'  ROC-AUC   : {results[name]["roc_auc"]:.4f}')
    print(f'  CV Acc    : {results[name]["cv_accuracy_mean"]:.4f} ± {results[name]["cv_accuracy_std"]:.4f}')
    print()

In [ ]:
results_df = pd.DataFrame(results).T.round(4)
results_df.style.background_gradient(cmap='YlGn', subset=['accuracy', 'f1_score', 'roc_auc'])

In [ ]:
plot_metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
results_df[plot_metrics].plot(kind='bar', figsize=(12, 6))
plt.title('Model Performance Comparison (All Metrics)')
plt.ylabel('Score')
plt.xlabel('Model')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Metric', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metric_labels = {
    'accuracy': 'Accuracy',
    'precision': 'Precision',
    'recall': 'Recall',
    'f1_score': 'F1-Score',
    'roc_auc': 'ROC-AUC',
    'cv_accuracy_mean': 'CV Accuracy (Mean)',
}
colors = sns.color_palette('Set2', len(results_df))

for ax, (metric, label) in zip(axes.flatten(), metric_labels.items()):
    bars = ax.bar(results_df.index, results_df[metric], color=colors)
    ax.set_title(label)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(results_df.index, rotation=30, ha='right')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# Hide the last (unused) subplot
axes.flatten()[-1].set_visible(False)

plt.suptitle('Individual Metric Comparison Across Models', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix heatmap for each model
n_models = len(models)
fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4))

for ax, (name, model) in zip(axes, trained_models.items()):
    y_pred = model.predict(X_test_selected)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Cancer', 'Cancer'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)

plt.suptitle('Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves for all models
fig, ax = plt.subplots(figsize=(8, 6))

for name, model in trained_models.items():
    y_prob = (
        model.predict_proba(X_test_selected)[:, 1]
        if hasattr(model, 'predict_proba')
        else model.decision_function(X_test_selected)
    )
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.500)')
ax.set_title('ROC Curves – All Models')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

In [ ]:
# Identify the best model by F1-Score (balances precision and recall).
best_by_f1 = results_df['f1_score'].idxmax()
best_by_acc = results_df['accuracy'].idxmax()
best_by_auc = results_df['roc_auc'].idxmax()

print('=== Best Model Summary ===')
print(f'  Best by Accuracy : {best_by_acc}  ({results_df.loc[best_by_acc, "accuracy"]:.4f})')
print(f'  Best by F1-Score : {best_by_f1}  ({results_df.loc[best_by_f1, "f1_score"]:.4f})')
print(f'  Best by ROC-AUC  : {best_by_auc}  ({results_df.loc[best_by_auc, "roc_auc"]:.4f})')
print()
print('Full metrics for the best F1-Score model:')
print(results_df.loc[best_by_f1].to_string())